# GovPulse: Premium Public Policy AI Toolkit
Notebook ini adalah versi mutakhir yang menggabungkan Automated Data Fetching, Advanced Analytics, dan Robust Model Validation untuk kebijakan publik yang akurat.

## Kerangka Strategis: Machine Learning dalam Tata Kelola
Penggunaan ML dalam kebijakan publik dibagi menjadi tiga fase utama:
1. **Fase Perencanaan**: Memahami masalah kompleks dan memprediksi dampak sebelum kebijakan dibuat.
2. **Fase Implementasi**: Memastikan operasional layanan publik (seperti klasifikasi aduan) berjalan otomatis dan adil.
3. **Fase Evaluasi**: Mengukur keberhasilan kebijakan melalui analisis sentimen dan audit kinerja anggaran.

## 1. Pengumpulan & Integrasi Data (Real-Time)
Mengambil dataset **World Happiness Report 2021** langsung dari sumber GitHub untuk memastikan data selalu aktual.

In [1]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split, cross_val_score, KFold
from sklearn.metrics import mean_absolute_error, r2_score

url = "https://raw.githubusercontent.com/michaelgreenacre/PCA/main/world-happiness-report-2021.csv"
df = pd.read_csv(url, encoding='utf-8-sig')
print(f"Data Berhasil Dimuat: {df.shape[0]} negara terpetakan.")
df.head()

Data Berhasil Dimuat: 149 negara terpetakan.


,Country name,Regional indicator,Ladder score,Standard error of ladder score,upperwhisker,lowerwhisker,Logged GDP per capita,Social support,Healthy life expectancy,Freedom to make life choices,Generosity,Perceptions of corruption,Ladder score in Dystopia,Explained by: Log GDP per capita,Explained by: Social support,Explained by: Healthy life expectancy,Explained by: Freedom to make life choices,Explained by: Generosity,Explained by: Perceptions of corruption,Dystopia + residual
0,Finland,Western Europe,7.842,0.032,7.904,7.780,10.775,0.954,72.0,0.949,-0.098,0.186,2.43,1.446,1.106,0.741,0.691,0.124,0.481,3.253
1,Denmark,Western Europe,7.620,0.035,7.687,7.552,10.933,0.954,72.7,0.946,0.030,0.179,2.43,1.502,1.108,0.763,0.686,0.208,0.485,2.868
2,Switzerland,Western Europe,7.571,0.036,7.643,7.500,11.117,0.942,74.4,0.919,0.025,0.292,2.43,1.566,1.079,0.816,0.653,0.204,0.413,2.839
3,Iceland,Western Europe,7.554,0.059,7.670,7.438,10.878,0.983,73.0,0.955,0.160,0.673,2.43,1.482,1.172,0.772,0.698,0.293,0.170,2.967
4,Netherlands,Western Europe,7.464,0.027,7.518,7.410,10.932,0.942,72.4,0.913,0.175,0.338,2.43,1.501,1.079,0.753,0.647,0.302,0.384,2.798


## 2. Pembersihan & Deteksi Outlier
Memastikan kualitas data melalui pengecekan nilai kosong dan distribusi ekstrim.

In [2]:
missing = df.isnull().sum().sum()
if missing > 0:
    df = df.dropna()
    
# Deteksi Skewness pada fitur ekonomi
skew = df['Logged GDP per capita'].skew()
print(f"Skewness GDP: {skew:.2f} (Nilai mendekati 0 berarti distribusi normal/bagus untuk model)")
print("Validasi Data: OK")

Skewness GDP: -0.35 (Nilai mendekati 0 berarti distribusi normal/bagus untuk model)
Validasi Data: OK


## 3. Visualisasi Geografis & Deskriptif
Menampilkan persebaran skor kebahagiaan di seluruh dunia menggunakan Peta Choropleth interaktif.

In [3]:
fig = px.choropleth(df, locations="Country name", locationmode='country names',
                    color="Ladder score", hover_name="Country name", 
                    color_continuous_scale=px.colors.sequential.Plasma,
                    title="Peta Kebahagiaan Dunia 2021 (Skala 0-10)")
fig.update_layout(height=600)
fig.show()

print(f"Skor Indonesia: {df[df['Country name']=='Indonesia']['Ladder score'].values[0]:.2f}")

C:\Users\LENOVO\AppData\Local\Temp\ipykernel_22676\1329664610.py:1: DeprecationWarning: The library used by the *country names* `locationmode` option is changing in an upcoming version. Country names in existing plots may not work in the new version. To ensure consistent behavior, consider setting `locationmode` to *ISO-3*.
  fig = px.choropleth(df, locations="Country name", locationmode='country names',


Skor Indonesia: 5.34


## 4. Feature Selection & Analysis
Memetakan indikator kebijakan sebagai variabel independen.

In [4]:
features = [
    'Logged GDP per capita', 'Social support', 'Healthy life expectancy', 
    'Freedom to make life choices', 'Generosity', 'Perceptions of corruption'
]
X = df[features]
y = df['Ladder score']

## 5. Robust Modeling & Validation (Fase Perencanaan & Evaluasi)
Tahap ini digunakan untuk memahami korelasi antar variabel kebijakan (**Evaluasi**) dan membangun model prediktif untuk masa depan (**Perencanaan**).

Menggunakan **K-Fold Cross-Validation** untuk memastikan akurasi tidak hanya tinggi secara kebetulan.

In [5]:
# Setup Cross-Validation
kf = KFold(n_splits=5, shuffle=True, random_state=42)

# 1. Random Forest (Baseline)
rf_pipe = Pipeline([('scaler', StandardScaler()), ('rf', RandomForestRegressor(n_estimators=100, random_state=42))])
rf_cv = cross_val_score(rf_pipe, X, y, cv=kf, scoring='r2')

# 2. Gradient Boosting (Advanced)
gb_pipe = Pipeline([('scaler', StandardScaler()), ('gb', GradientBoostingRegressor(n_estimators=100, random_state=42))])
gb_cv = cross_val_score(gb_pipe, X, y, cv=kf, scoring='r2')

print(f"R2 Score (Random Forest): {rf_cv.mean():.4f} (+/- {rf_cv.std():.4f})")
print(f"R2 Score (Gradient Boosting): {gb_cv.mean():.4f} (+/- {gb_cv.std():.4f})")

# Pilih model terbaik
final_model = gb_pipe.fit(X, y)
print("Model divalidasi dan siap digunakan.")

R2 Score (Random Forest): 0.7328 (+/- 0.0777)
R2 Score (Gradient Boosting): 0.7389 (+/- 0.0599)
Model divalidasi dan siap digunakan.


## 5.1. Analisis Residual (Audit Kinerja Model)
Digunakan untuk memastikan model bekerja secara adil dan tidak bias terhadap kelompok negara tertentu, sejalan dengan audit kinerja tata kelola digital.

In [6]:
y_all_pred = final_model.predict(X)
residuals = y - y_all_pred

fig_res = px.scatter(x=y_all_pred, y=residuals, 
                     labels={'x': 'Predicted Value', 'y': 'Residual (Error)'},
                     title="Analisis Residual: Mendeteksi Bias Model")
fig_res.add_hline(y=0, line_dash="dash", line_color="red")
fig_res.show()

## 6. Simulator Kebijakan Preskriptif (Fase Perencanaan & Implementasi)
Gunakan tool ini untuk mensimulasikan dampak sebelum kebijakan benar-benar dijalankan. Ini mencegah kesalahan sasaran dan anggaran dalam perencanaan pembangunan.

In [7]:
def policy_simulation(country_name, social_boost, corruption_reduction):
    current = df[df['Country name'] == country_name][features].copy()
    baseline = final_model.predict(current)[0]
    
    scenario = current.copy()
    scenario['Social support'] = np.clip(scenario['Social support'] + social_boost, 0, 1)
    scenario['Perceptions of corruption'] = np.clip(scenario['Perceptions of corruption'] - corruption_reduction, 0, 1)
    
    pred = final_model.predict(scenario)[0]
    print(f"--- Simulasi Kebijakan: {country_name} ---")
    print(f"Baseline: {baseline:.2f} | Proyeksi: {pred:.2f} | Kenaikan: {pred-baseline:.2f}")
    return baseline, pred

policy_simulation('Indonesia', social_boost=0.1, corruption_reduction=0.1)

--- Simulasi Kebijakan: Indonesia ---
Baseline: 5.32 | Proyeksi: 5.56 | Kenaikan: 0.23


(np.float64(5.3240527023707855), np.float64(5.5551166868681925))

## 🔬 Studi Kasus: Krisis Biaya Hidup (Simulasi Perencanaan)

Mendemonstrasikan **Fase Perencanaan** di mana pemerintah mensimulasikan dampak kenaikan harga barang (inflasi) dan mencari solusi bantalan sosial (Social Safety Net) yang paling efektif.

In [8]:
def simulasi_krisis_inflasi(country_name, inflation_rate=0.10):
    """
    Simulasi krisis: Penurunan daya beli (GDP) akibat inflasi.
    """
    # 1. Data Baseline
    current = df[df['Country name'] == country_name][features].copy()
    baseline_pred = final_model.predict(current)[0]
    
    # 2. Skenario Krisis (GDP turun karena daya beli melemah)
    crisis = current.copy()
    crisis['Logged GDP per capita'] *= (1 - inflation_rate)
    crisis_pred = final_model.predict(crisis)[0]
    
    # 3. Intervensi Kebijakan (Cari tambahan Social Support yang dibutuhkan)
    # Kita coba naikkan social support sampai kebahagiaan kembali ke baseline
    intervention = crisis.copy()
    required_boost = 0.0
    current_pred = crisis_pred
    
    while current_pred < baseline_pred and intervention['Social support'].values[0] < 1.0:
        required_boost += 0.01
        intervention['Social support'] = np.clip(current.values[0][1] + required_boost, 0, 1)
        current_pred = final_model.predict(intervention)[0]
        if required_boost > 0.5: break # Limit
        
    print(f"=== Analisis Strategis Krisis Biaya Hidup ({country_name}) ===")
    print(f"1. Prediksi Awal (Normal)     : {baseline_pred:.2f}")
    print(f"2. Prediksi Pasca Inflasi {inflation_rate*100:.0f}%: {crisis_pred:.2f} (Penurunan: {baseline_pred-crisis_pred:.2f})")
    print(f"3. Rekomendasi Solusi         : Booster 'Social Support' sebesar +{required_boost:.2f}")
    print(f"4. Hasil Akhir (Intervensi)   : {current_pred:.2f}")
    
    # Visualisasi Skenario
    scenarios = ['Baseline', f'Crisis ({inflation_rate*100:.0f}%)', 'With Policy Response']
    scores = [baseline_pred, crisis_pred, current_pred]
    fig = px.bar(x=scenarios, y=scores, color=scenarios, 
                 title=f"Strategi Mitigasi Krisis Biaya Hidup: {country_name}",
                 labels={'x': 'Skenario Kebijakan', 'y': 'Happiness Index'})
    fig.show()

# Jalankan simulasi untuk Indonesia dengan asumsi inflasi 10%
simulasi_krisis_inflasi('Indonesia', inflation_rate=0.10)

=== Analisis Strategis Krisis Biaya Hidup (Indonesia) ===
1. Prediksi Awal (Normal)     : 5.32
2. Prediksi Pasca Inflasi 10%: 5.23 (Penurunan: 0.10)
3. Rekomendasi Solusi         : Booster 'Social Support' sebesar +0.02
4. Hasil Akhir (Intervensi)   : 5.33


## Kesimpulan Berbasis Siklus Kebijakan

1. **Fase Perencanaan (Simulasi Dampak)**: Melalui fitur Policy Simulation, kita dapat memproyeksikan kenaikan kebahagiaan publik sebelum intervensi dilakukan.
2. **Fase Implementasi (Otomasi & Efisiensi)**: Model terlatih dapat diintegrasikan ke dalam backend GovPulse untuk merespons aduan atau mengaudit data bansos secara otomatis.
3. **Fase Evaluasi (Audit & Sentimen)**: Melalui transparansi model (Feature Importance), kita mengetahui apakah variabel ekonomi (GDP) atau sosial benar-benar berkorelasi dengan hasil di lapangan.

In [9]:
importances = final_model.named_steps['gb'].feature_importances_
feat_df = pd.DataFrame({'Indikator': features, 'Kepentingan': importances}).sort_values('Kepentingan')
px.bar(feat_df, x='Kepentingan', y='Indikator', orientation='h', title="Transparansi: Faktor Penentu Global").show()